In [4]:
import os
import xml.etree.ElementTree as ET
import pandas as pd

folder_path = r"C:\Users\rijum\Downloads\End_Sem_Model\MedQuAD-master"

data = []
for root_dir, dirs, files in os.walk(folder_path):
    for file in files:
        if file.endswith(".xml"):
            try:
                tree = ET.parse(os.path.join(root_dir, file))
                root = tree.getroot()
                for qa in root.findall('.//QAPair'):
                    q, a = qa.find('Question'), qa.find('Answer')
                    if q is not None and a is not None:
                        data.append([q.text, a.text])
            except: pass

df = pd.DataFrame(data, columns=["question", "answer"])

# Remove any rows that have *any* missing values.
df.dropna(inplace=True)

# Discard rows where the *question* column is an empty string after stripping.
df = df[df["question"].str.strip() != ""]

# Discard rows where the *answer* column is an empty string after stripping.
df = df[df["answer"].str.strip() != ""]

# Trim leading/trailing whitespace from all questions.
df["question"] = df["question"].str.strip()

# Trim leading/trailing whitespace from all answers.
df["answer"]   = df["answer"].str.strip()

# Remove duplicate *questions* while keeping the first occurrence.
df.drop_duplicates(subset=["question"], inplace=True)

# Reset the index after all the filtering operations.
df.reset_index(drop=True, inplace=True)
print(f"Clean QA pairs loaded: {len(df)}")

Clean QA pairs loaded: 14979


In [5]:
from langchain_core.documents import Document

documents = [
    Document(page_content=f"Medical Question: {row['question']}\nMedical Answer: {row['answer']}")
    for _, row in df.iterrows()
]
print(f"Total documents: {len(documents)}")

Total documents: 14979


In [6]:
# No chunking
chunks = documents
print(f"Total documents (no chunking): {len(chunks)}")

Total documents (no chunking): 14979


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"batch_size": 64}
)

print("Building FAISS index...")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("medquad_index")
print("FAISS index built and saved!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Building FAISS index...
FAISS index built and saved!


In [8]:
from dotenv import load_dotenv
import os
load_dotenv()  # loads .env file
# Load HF token from .env
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [9]:
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"batch_size": 64}
)

vectorstore = FAISS.load_local(
    "medquad_index", embeddings,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(
    search_type="mmr",                          
    search_kwargs={"k": 6, "fetch_k": 20}       
)

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile", 
    temperature=0.1
)

prompt = PromptTemplate.from_template("""You are a friendly and easy-to-understand health assistant.
Use ONLY the information provided in the context below to answer. Do NOT use outside knowledge.

IMPORTANT RULES:
- Always use the simple, everyday common name of the disease (e.g., say 'High Blood Sugar' instead of 'Type 2 Diabetes Mellitus', 'High Blood Pressure' instead of 'Hypertension', 'Weak Heart' instead of 'Congestive Heart Failure').
- Focus on what condition the person likely HAS RIGHT NOW based on their symptoms — NOT what they might develop or die from in the future.
- Write as if you are explaining to a friend with no medical background. Use short, simple sentences.
- Do NOT use scary or death-related language.
- Do NOT use medical abbreviations without first explaining them in plain words.

Answer strictly in this format:

**Most Likely Condition:**
<Write the simple, everyday common name. Example: 'High Blood Sugar (Diabetes)' or 'Chest Infection (Pneumonia)'>

**What This Means (In Simple Words):**
<In 2-3 easy sentences, explain what this condition is and how the person's symptoms match it. Use simple language that a school student can understand.>

**Go See a Doctor If:**
- <Warning sign 1>
- <Warning sign 2>
- <Warning sign 3>

**What You Can Do Next:**
Please visit the 'Medicine Recommender' section of this app for personalized medicine suggestions, diet tips, and safety precautions for your condition.

If the context does not have enough information to answer, say exactly:
"I'm not sure based on the available information. Please see a doctor for a proper check-up."

Context:
{context}

Question:
{question}

Answer:""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {
        "context": lambda x: format_docs(retriever.invoke(x)),
        "question": lambda x: x
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready!")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RAG chain ready!


In [10]:
query = """I am a 60-year-old male with Type 2 diabetes, hypertension, and chronic kidney disease. 
I recently started experiencing chest pain, shortness of breath, swollen ankles, 
and blurred vision. I also have a family history of heart disease and stroke. 
What conditions could be responsible for these symptoms, what are the treatment options, 
and what precautions should I take to avoid further complications?"""

print("Generating response...\n")
for chunk in qa_chain.stream(query):
    print(chunk, end="", flush=True)

Generating response...

**Most Likely Condition:**
Weak Heart

**What This Means (In Simple Words):**
You might have a weak heart because of your chest pain, shortness of breath, and swollen ankles. These symptoms can happen when your heart is not working well. Your diabetes, high blood pressure, and kidney disease can also affect your heart. Your family history of heart disease and stroke is another reason to think about your heart health.

**Go See a Doctor If:**
- You have chest pain that doesn't go away
- You feel short of breath even when you're resting
- Your ankles are very swollen
- You have blurred vision that doesn't get better

**What You Can Do Next:**
Please visit the 'Medicine Recommender' section of this app for personalized medicine suggestions, diet tips, and safety precautions for your condition.

In [11]:
query = """I have been feeling incredibly thirsty lately, no matter how much water I drink. 
I also have to pee all the time, especially in the middle of the night. 
On top of that, I feel exhausted all day, and I noticed a small cut on my foot is taking weeks to heal. 
What could be wrong with me?"""

print("Generating response...\n")
for chunk in qa_chain.stream(query):
    print(chunk, end="", flush=True)

Generating response...

**Most Likely Condition:**
High Blood Sugar (Diabetes)

**What This Means (In Simple Words):**
You might have a condition where your body has too much sugar in the blood. This can make you feel very thirsty and need to pee a lot. The exhaustion and slow healing of your cut could also be related to this condition. It's like your body is not working properly to keep you healthy and full of energy.

**Go See a Doctor If:**
- You feel extremely weak or tired
- You have blurry vision or other vision problems
- You notice other cuts or wounds taking a long time to heal

**What You Can Do Next:**
Please visit the 'Medicine Recommender' section of this app for personalized medicine suggestions, diet tips, and safety precautions for your condition.